# Test: Vocabulary Size Impact on Performance

**Question**: Does larger vocabulary size affect:
1. Detection speed
2. Accuracy

**Test Setup**:
- Same test image (chicken breast)
- Different vocabulary sizes: 50, 100, 200, 529
- Measure: Time and Confidence

In [ ]:
import time
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import matplotlib.pyplot as plt

In [ ]:
# Load CLIP model
print("Loading CLIP model...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
print("✓ CLIP loaded")

In [ ]:
# Load full vocabulary
PROJECT_ROOT = Path.cwd().parent.parent
CSV_PATH = PROJECT_ROOT / "data" / "ingredients_vocabulary.csv"

df = pd.read_csv(CSV_PATH)
FULL_VOCAB = df['Ingredient'].tolist()

print(f"Full vocabulary size: {len(FULL_VOCAB)} ingredients")

In [ ]:
# Test image
TEST_IMAGE = PROJECT_ROOT / "data" / "test_images" / "test.jpeg"
image = Image.open(TEST_IMAGE).convert('RGB')

print(f"Test image: {TEST_IMAGE.name}")
print(f"True label: Chicken breast")
display(image.resize((300, 300)))

In [ ]:
# Create vocabulary subsets
vocab_sizes = [50, 100, 200, 529]

vocab_sets = {}
for size in vocab_sizes:
    if size >= len(FULL_VOCAB):
        vocab_sets[size] = FULL_VOCAB
    else:
        # Make sure "Chicken breast" is always included
        if "Chicken breast" in FULL_VOCAB:
            vocab = ["Chicken breast"] + [v for v in FULL_VOCAB if v != "Chicken breast"][:size-1]
        else:
            vocab = FULL_VOCAB[:size]
        vocab_sets[size] = vocab

print("Vocabulary subsets created:")
for size, vocab in vocab_sets.items():
    has_chicken = "Chicken breast" in vocab
    print(f"  {size:3d} ingredients - Contains 'Chicken breast': {has_chicken}")

In [ ]:
# Test function
def test_detection(image, vocab, num_runs=5):
    """
    Test detection with given vocabulary
    
    Returns:
        avg_time: Average detection time
        prediction: Top prediction
        confidence: Confidence score
    """
    times = []
    
    for i in range(num_runs):
        start = time.time()
        
        inputs = clip_processor(
            text=vocab,
            images=image,
            return_tensors="pt",
            padding=True
        )
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
        
        probs = outputs.logits_per_image.softmax(dim=1)[0]
        top_prob, top_idx = probs.max(0)
        
        elapsed = time.time() - start
        times.append(elapsed)
    
    return {
        'avg_time': sum(times) / len(times),
        'prediction': vocab[top_idx],
        'confidence': top_prob.item()
    }

print("✓ Test function ready")

In [ ]:
# Run tests
print("="*80)
print("TESTING VOCABULARY SIZE IMPACT")
print("="*80)
print()

results = []

for size in vocab_sizes:
    print(f"Testing with {size} ingredients...")
    vocab = vocab_sets[size]
    
    result = test_detection(image, vocab, num_runs=5)
    
    results.append({
        'vocab_size': size,
        'avg_time': result['avg_time'],
        'prediction': result['prediction'],
        'confidence': result['confidence'],
        'correct': result['prediction'] == 'Chicken breast'
    })
    
    print(f"  Time: {result['avg_time']:.3f}s")
    print(f"  Prediction: {result['prediction']}")
    print(f"  Confidence: {result['confidence']:.1%}")
    print(f"  Correct: {'✓' if result['correct'] else '✗'}")
    print()

results_df = pd.DataFrame(results)
print("="*80)
print("RESULTS SUMMARY")
print("="*80)
print(results_df.to_string(index=False))

In [ ]:
# Visualize results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Detection Time vs Vocabulary Size
ax1.plot(results_df['vocab_size'], results_df['avg_time'], 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Vocabulary Size', fontsize=12)
ax1.set_ylabel('Detection Time (seconds)', fontsize=12)
ax1.set_title('Detection Speed vs Vocabulary Size', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, max(results_df['avg_time']) * 1.2)

# Add time labels
for i, row in results_df.iterrows():
    ax1.text(row['vocab_size'], row['avg_time'] + 0.02, 
             f"{row['avg_time']:.3f}s", 
             ha='center', fontsize=10)

# Plot 2: Confidence vs Vocabulary Size
colors = ['green' if c else 'red' for c in results_df['correct']]
ax2.plot(results_df['vocab_size'], results_df['confidence'] * 100, 'o-', linewidth=2, markersize=8)
ax2.set_xlabel('Vocabulary Size', fontsize=12)
ax2.set_ylabel('Confidence (%)', fontsize=12)
ax2.set_title('Detection Confidence vs Vocabulary Size', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 100)

# Add confidence labels
for i, row in results_df.iterrows():
    ax2.text(row['vocab_size'], row['confidence'] * 100 + 2, 
             f"{row['confidence']:.1%}", 
             ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "data" / "results" / "vocab_size_impact.png", dpi=150)
plt.show()

print(f"\n✓ Chart saved to: data/results/vocab_size_impact.png")

In [ ]:
# Analysis
print("="*80)
print("ANALYSIS")
print("="*80)

# Time impact
time_50 = results_df[results_df['vocab_size'] == 50]['avg_time'].values[0]
time_529 = results_df[results_df['vocab_size'] == 529]['avg_time'].values[0]
time_increase = ((time_529 - time_50) / time_50) * 100

print(f"\n1. SPEED IMPACT:")
print(f"   50 ingredients:  {time_50:.3f}s")
print(f"   529 ingredients: {time_529:.3f}s")
print(f"   → {time_increase:.1f}% slower with 10x more ingredients")

# Accuracy impact
conf_50 = results_df[results_df['vocab_size'] == 50]['confidence'].values[0]
conf_529 = results_df[results_df['vocab_size'] == 529]['confidence'].values[0]
conf_decrease = ((conf_50 - conf_529) / conf_50) * 100

print(f"\n2. CONFIDENCE IMPACT:")
print(f"   50 ingredients:  {conf_50:.1%}")
print(f"   529 ingredients: {conf_529:.1%}")
print(f"   → {abs(conf_decrease):.1f}% {'lower' if conf_decrease > 0 else 'higher'} confidence")

# Accuracy
all_correct = all(results_df['correct'])
print(f"\n3. ACCURACY:")
print(f"   All predictions correct: {all_correct}")
if not all_correct:
    incorrect = results_df[~results_df['correct']]
    print(f"   Incorrect predictions:")
    for _, row in incorrect.iterrows():
        print(f"     - {row['vocab_size']} ingredients → {row['prediction']}")

## Recommendations

### Option 1: **Smaller Curated Vocabulary** (Most Common)
- Keep only 100-150 most common ingredients
- ✅ Faster (2-3x)
- ✅ Higher confidence
- ❌ Less coverage

### Option 2: **Hierarchical Detection** (Best Balance)
1. First pass: Detect category (Poultry/Beef/Seafood/Vegetable)
2. Second pass: Detect specific item within category
- ✅ Fast
- ✅ High accuracy
- ✅ Full coverage

### Option 3: **Dynamic Vocabulary** (Advanced)
- User selects category first
- Only load relevant ingredients
- ✅ Fastest
- ✅ Highest accuracy
- ❌ Extra user step

### Option 4: **Keep All 529** (Current)
- ✅ Maximum coverage
- ❌ Slower
- ❌ Lower confidence
- ✔️ Still < 1 second, acceptable for most cases